# Loyiha 5: Capstone — to'liq ML pipeline va GitHub portfolio — Telco Customer Churn (IBM)

**Talaba:** student3 · **Modul:** M0–M4 (hammasi) · **Kitob (ML_GUIDE):** 1-bob (Workflow, Train/Val/Test), 2-bob (Pipeline, ColumnTransformer) + 4–7-boblar · **Qiyinlik:** ⭐⭐⭐⭐⭐ · **Taxminiy vaqt:** 12–16 soat · **Muddat:** ____________
> **Qoidalar.** Bu notebook — sizning ish daftaringiz. Har bir `# TODO` katakni to'ldiring, har bir **Savol** ga darhol pastida yangi *markdown* katak ochib 2–5 gap bilan javob bering ("ha/yo'q" baholanmaydi). Topshirishdan oldin `Kernel → Restart & Run All` — notebook yuqoridan pastga **xatosiz** ishlashi shart. Internetdan o'rganish — mumkin va kerak; tayyor kodni nusxalash — yo'q (himoyada har bir qatorni tushuntirib bera olishingiz kerak).

## Loyiha haqida

Bu — **portfolio loyiha**. Uni GitHub ga yuklaysiz, README ni ish beruvchi o'qiydi. Farq: oldingi loyihalarda ma'lumot "tayyor" edi; bu yerda matnli ustunlar, missing values, iflos qiymatlar, leakage tuzoqlari bor — hammasi kitob 2-bobdagi **leakage-siz** `Pipeline + ColumnTransformer` bilan yechiladi.

**Nimani o'rganasiz:** muammo bayoni → tozalash → EDA → feature engineering → `ColumnTransformer` (numeric: impute+scale; categorical: impute+one-hot) → baseline → LogisticRegression (C, L1/L2, class_weight) → CV bilan tanlash → test **bir marta** → interpretatsiya → xato tahlili → `joblib` bilan saqlash → `predict()` funksiyasi → README.

**Qoida:** test to'plamiga **faqat oxirida, bir marta** qaraysiz. Barcha tanlovlar (model, alpha/C, feature'lar) — train + cross-validation bilan.

## Dataset: Telco Customer Churn (IBM)

7043 ta telekom mijozi: demografiya, xizmatlar (internet, TV, telefon), shartnoma turi, to'lov usuli, oylik va umumiy to'lov → **mijoz ketdimi** (Churn, 27%). Har bir telekom kompaniyasining asosiy ML masalasi: ketishi mumkin bo'lgan mijozga oldindan taklif berish yangi mijoz topishdan 5× arzon.

**Manba:** https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv (IBM sample dataset)

| Ustun | Ma'nosi |
|---|---|
| customerID | identifikator — olib tashlang |
| gender, SeniorCitizen (0/1), Partner, Dependents | demografiya |
| tenure | mijoz bo'lgan oylar (0–72) |
| PhoneService, MultipleLines, InternetService, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies | xizmatlar (Yes / No / "No internet service") |
| Contract, PaperlessBilling, PaymentMethod | Month-to-month / One year / Two year; to'lov usuli |
| MonthlyCharges | oylik to'lov, $ |
| TotalCharges | umumiy to'lov — **matn (object)!** 11 ta bo'sh qiymat |
| **Churn** | Yes / No — target (26.5% Yes) |

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, joblib
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)                          # 7043 mijoz
TARGET = 'Churn'
print(df.shape); df.head()

## 1-vazifa: Muammo bayoni (markdown, 1 paragraf)
Kim bu bashoratdan foydalanadi? Qaror qanday o'zgaradi? Qaysi metrika **biznes uchun** muhim va nega? Modelning xatosi kimga zarar keltiradi?

Kontekst: retention bo'limi har oy 500 ta mijozga chegirma taklif qila oladi. Model "ketadi" degan mijozlar ro'yxatini beradi. **False negative** = ketadigan mijozni o'tkazib yuborish (yo'qotilgan daromad ~ 12 × MonthlyCharges); **false positive** = qolmoqchi bo'lgan mijozga bekorga chegirma. Qaysi metrika: recall? precision@500? F1? O'zbekiston operatorlari (Beeline, Ucell, Uzmobile) uchun qaysi ustunlar mavjud bo'lardi?

**Muammo bayoni:** _(shu yerga yozing)_

## 2-vazifa: Tozalash va leakage tekshiruvi
**Tuzoqlar:** (1) `TotalCharges` — `object` dtype, 11 ta bo'sh string (`' '`) — bular `tenure == 0` yangi mijozlar. `pd.to_numeric(errors='coerce')` → NaN → pipeline ichida impute (yoki 0 — asoslang). (2) `customerID` olib tashlang. (3) `SeniorCitizen` 0/1 raqam, lekin kategorik — qanday muomala qilasiz? (4) "No internet service" — bu 6 ta ustunda takrorlanadi, `InternetService == 'No'` bilan bir xil ma'lumot — dublikat. Leakage bormi? `TotalCharges ≈ tenure × MonthlyCharges` — bu leakage emas, lekin kuchli collinearity.

**Savol 2.** Qaysi ustunlar **leakage** (target haqida "kelajakdan" ma'lumot beradi yoki targetning o'zi boshqa ko'rinishda)? Ularni nega olib tashlash shart? Kitob 1-bob "Common mistakes" ga havola bering.

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.drop(columns='customerID')
# TODO: Churn → 0/1, dtype tekshiruvi
# TODO: tozalash, leakage ustunlarini olib tashlash

## 3-vazifa: EDA (kamida 6 grafik, har biri ostida 1 gap xulosa)
Target taqsimoti · numeric feature'lar vs target · kategorik feature'lar vs target (`groupby().mean()` bar chart) · korrelyatsiya heatmap · missing values xaritasi (`df.isna().mean()`) · siz tanlagan bitta "qiziq" grafik.

**Savol 3.** EDA dan **3 ta gipoteza** yozing ("… bo'lsa target … bo'ladi"). Ularni keyin model koeffitsiyentlari bilan tekshirasiz (8-vazifa).

In [ ]:
# TODO: EDA — 6+ grafik

## 4-vazifa: Feature engineering (kamida 2 ta yangi feature)
G'oyalar: `n_services` — mijoz ulangan xizmatlar soni (Yes larni sanash); `charge_ratio = TotalCharges / (tenure × MonthlyCharges)` (narx o'zgarganini ko'rsatadi; tenure=0 ga ehtiyot bo'ling); `tenure_group` (0–12 / 13–36 / 37+); `is_month_to_month` × `fiber` interaction (EDA: Month-to-month + Fiber optic — churn eng yuqori). Kamida 2 tasini quring.

Har bir yangi feature uchun 1 gap **sabab**. Split dan **oldin** faqat qator-ichi (row-wise) hisoblar ruxsat; agregatlar (o'rtacha, target bo'yicha) — faqat pipeline ichida.

**Savol 4.** Yangi feature'laringiz target bilan korrelyatsiyasi asl feature'lardan yuqorimi? "Target encoding" nega split dan oldin qilinsa leakage bo'ladi?

In [ ]:
# TODO: yangi feature'lar

## 5-vazifa: Split va `ColumnTransformer` + `Pipeline`
`train_test_split(test_size=0.2, random_state=42, stratify=y)`. Preprocessor:
- numeric → `SimpleImputer(strategy='median')` → `StandardScaler()`
- categorical → `SimpleImputer(strategy='most_frequent')` → `OneHotEncoder(handle_unknown='ignore')`

`Pipeline([('prep', preprocessor), ('model', ...)])`. `preprocessor.fit_transform(X_train).shape` ni chiqaring.

**Savol 5.** `handle_unknown='ignore'` nega kerak (production da nima bo'lishi mumkin)? Imputer ni pipeline ichida qilish bilan `df.fillna(df.median())` ni split dan oldin qilish farqi nimada (kitob 2-bob)?

In [ ]:
num_cols = [...]  # TODO
cat_cols = [...]  # TODO
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
])
# TODO: split, Pipeline, shape

## 6-vazifa: Baseline → Logistic Regression → sozlash (faqat train + CV)
1. `DummyClassifier(strategy='most_frequent')` — CV accuracy, F1.
2. `LogisticRegression(max_iter=5000)` — `cross_val_score` bilan accuracy **va** F1 (`scoring='f1'`), `cv=5`.
3. `GridSearchCV`: `model__C ∈ np.logspace(-3, 2, 11)`, `model__penalty ∈ ['l1', 'l2']` (`solver='liblinear'`), `model__class_weight ∈ [None, 'balanced']`, `scoring='f1'` (yoki 1-vazifada tanlagan metrikangiz). `cv_results_` dan C vs F1 grafigi.
4. `Contract` ni olib tashlab model quring — F1 qanchaga tushadi (ablation)? EDA da Month-to-month churn 43%, Two year 3% — model buni "o'rgandi" yoki shunchaki yodladimi?

**Savol 6.** Baseline accuracy qancha? Logistic Regression undan qanchalik yaxshi — accuracy da va F1 da? Qaysi biri "haqiqiy" yaxshilanishni ko'rsatadi? `class_weight='balanced'` nima qildi?

In [ ]:
# TODO: baseline, CV, GridSearchCV, grafik

## 7-vazifa: Test — bir marta
Eng yaxshi pipeline ni **butun train** ga fit qiling, test da: accuracy, precision, recall, F1, ROC-AUC, confusion matrix heatmap. Jadval: Baseline · oddiy model · eng yaxshi model.

**Savol 7.** Test natijasi CV natijasidan farq qildimi? Farq katta bo'lsa nima degani? Endi "yana biroz sozlab test da qayta ko'rish" nega **mumkin emas**?

In [ ]:
# TODO: final fit, test metrikalari, jadval

## 8-vazifa: Interpretatsiya
🔍 `pipeline.named_steps['prep'].get_feature_names_out()` bilan one-hot dan keyingi feature nomlarini oling. Koeffitsiyentlarni nomlar bilan bog'lang → eng katta 10 musbat va 10 manfiy — gorizontal bar chart. 3-savoldagi gipotezalaringizni tekshiring.

**Savol 8.** Koeffitsiyentlar scale qilingan feature'lar uchun — demak ularni **solishtirish mumkin**; nega? One-hot ustunlar koeffitsiyentini qanday o'qiysiz ("… kategoriyasi tashlab ketilgan kategoriyaga nisbatan …")? Qaysi gipotezangiz tasdiqlanmadi?

In [ ]:
# TODO: get_feature_names_out, top ±10 koeffitsiyent

## 9-vazifa: Xato tahlili
Test dagi barcha **false negative** va **false positive** larni ko'ring: ular feature'lar bo'yicha to'g'ri bashorat qilinganlardan qanday farq qiladi (`groupby` bilan o'rtachalar)?

**Savol 9.** Model qaysi "turdagi" namunalarda yomon ishlaydi? Buni tuzatish uchun **ma'lumot** kerakmi, **feature** kerakmi, yoki **boshqa model** kerakmi — asoslang.

In [ ]:
# TODO: xato tahlili

## 10-vazifa: Saqlash va `predict()` funksiyasi
`joblib.dump(best_pipeline, 'model.joblib')`. Funksiya: `predict_one(row: dict) -> float/str` — dict dan `pd.DataFrame([row])` yasab, pipeline orqali bashorat. 2 ta o'zingiz o'ylab topgan realistik misol bilan chaqiring (bittasi "chekka" holat — masalan yangi/ko'rilmagan kategoriya).

**Savol 10.** Nega **butun pipeline** (scaler + encoder + model) saqlanadi, faqat model emas? Ko'rilmagan kategoriya berilganda nima bo'ldi (5-savol)?

In [ ]:
# TODO: joblib, predict_one, 2 misol

## 11-vazifa: GitHub va README
Repo strukturasi:
```
telco-churn-prediction/
├── README.md            # ingliz tilida (portfolio) — quyidagi bo'limlar
├── student3_P5.ipynb   # shu notebook
├── model.joblib
├── requirements.txt     # pip freeze emas — faqat kerakli 6-8 kutubxona, versiya bilan
└── data/README.md       # dataset qayerdan olinadi (faylning o'zi emas, agar > 10 MB)
```
README bo'limlari: Overview (2 gap) · Dataset · Approach (pipeline diagrammasi so'z bilan) · **Results jadvali** (baseline vs best, test metrikalari) · Key findings (3–5 bullet, koeffitsiyentlardan) · How to run · Limitations & next steps.

Repo havolasini notebook oxiriga yozing. `.gitignore` da katta fayllar/venv bo'lsin.

## 12-vazifa: Xulosa (o'zbek tilida, 10–15 gap)
Natija · eng muhim 3 topilma · modelning chegaralari (nima uchun ishonib bo'lmaydi) · agar 1 hafta ko'proq vaqt bo'lsa nima qilardingiz.

## 🔍 Tadqiqot (majburiy — kamida 2 tasi)
1. **Threshold va biznes**: F1 ni maksimallashtiruvchi threshold ni CV predict_proba lar (`cross_val_predict`) bilan toping — test ga tegmasdan. Test da 0.5 va tanlangan threshold ni solishtiring.
2. **Learning curve** eng yaxshi pipeline uchun: ko'proq ma'lumot yordam beradimi?
3. **Adolat (fairness)**: Recall ni `SeniorCitizen` va `gender` bo'yicha alohida hisoblang. Keksa mijozlar uchun model yomonroq ishlaydimi? Agar retention taklifi faqat model tanlaganlarga berilsa — qaysi guruh sistemali "e'tibordan chetda" qoladi?
4. **Precision@k.** Biznes 500 ta mijozga taklif bera oladi: test da (yakuniy baholash — tanlov emas) `predict_proba` bo'yicha saralab, eng yuqori 500 (yoki test hajmiga mos 10%) ichida haqiqiy churn ulushi qancha? Tasodifiy 500 bilan solishtiring (lift). Bu metrika F1 dan nega biznesga yaqinroq?

In [ ]:
# TODO: tadqiqot

## ⭐ Bonus (+10)
**Streamlit** mini-ilova: mijoz ma'lumotlarini forma orqali kiritish → churn ehtimoli + "nega" (eng katta 3 ta koeffitsiyent × feature qiymati). Repo ga `app.py` qo'shing va README da screenshot ko'rsating.

## Topshirish

- Fayl nomi: `student3_P5.ipynb` (shu fayl, to'ldirilgan). `Restart & Run All` qilingan, barcha grafiklar ko'rinib turibdi.
- Oxirgi katak — **Xulosa** (markdown, 6–10 gap): nimani o'rgandingiz, qaysi natija kutilmagan bo'ldi, nimani hali tushunmadingiz.
- Himoya: 5 daqiqa — istalgan katakdagi kodni tushuntirib berasiz.
- **GitHub repo havolasi** — majburiy. README ingliz tilida, xulosa o'zbek tilida.

## Baholash (100 ball + bonus)

| Mezon | Ball |
|---|---|
| Tozalash + leakage + EDA + feature engineering (2–4) | 20 |
| Pipeline/ColumnTransformer to'g'ri, leakage yo'q, test bir marta (5–7) | 25 |
| Interpretatsiya + xato tahlili + predict() (8–10) | 15 |
| Savollarga tahliliy javoblar (10 ta) + Xulosa | 15 |
| GitHub repo + README sifati | 15 |
| Tadqiqot (2 ta) | 10 |
| ⭐ Bonus | +10 |